In [ ]:
import random

import os
import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import os
from sklearn.model_selection import train_test_split
from internal.persistence_manager import PersistenceManager
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import ColorJitter
import gc
from sklearn.model_selection import StratifiedKFold
import torch.nn.functional as F
from torchvision import models
from torch import nn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, classification_report
)


# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader

# Configurazione di TensorBoard e directory
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
import copy
import shutil
from itertools import product
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.gridspec as gridspec
from concurrent.futures import ThreadPoolExecutor, Future
from tqdm import tqdm
from PIL.ImageFile import ImageFile
from sklearn.model_selection import StratifiedKFold
from internal.persistence_manager import PersistenceManager

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline
## Load data
from internal.persistence_manager import PersistenceManager

data = PersistenceManager.load_dataset()
train_df = data.train_df
test_df = data.test_df

# Transform + patching
# Patch size used for both training and inference.
# Large enough to capture context, but still manageable in terms of memory.
PATCH_SIZE = 448

# Data augmentation applied only during training.
# The goal is to improve generalization without altering semantics too much.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=(-90, 90)),  # limited rotation range
    transforms.ColorJitter(0.15, 0.15, 0.15, 0.02), # mild color and brightness variations
    transforms.ToTensor(),
    transforms.Normalize(                          # ImageNet normalization (ConvNeXt pretrained)
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Validation / test transforms: no randomness, only normalization
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

def random_patch_pil(img: Image.Image, patch_size: int):
    """
    Extracts a single random square patch from an image.
    If the image is smaller than the patch size, it is padded first.
    Used during training to introduce spatial randomness.
    """
    w, h = img.size

    # Pad image if it is smaller than the desired patch size
    if w < patch_size or h < patch_size:
        pad_w = max(0, patch_size - w)
        pad_h = max(0, patch_size - h)
        img = transforms.functional.pad(img, padding=(0, 0, pad_w, pad_h), fill=0)
        w, h = img.size

    # Random top-left corner for the crop
    x = random.randint(0, w - patch_size)
    y = random.randint(0, h - patch_size)

    return img.crop((x, y, x + patch_size, y + patch_size))


def grid_patches_pil(img: Image.Image, patch_size: int, stride: int):
    """
    Extracts a grid of overlapping patches from an image.
    Ensures full spatial coverage, including image borders.
    Used at test time to aggregate predictions over the whole image.
    """
    w, h = img.size

    # Pad image if needed
    if w < patch_size or h < patch_size:
        pad_w = max(0, patch_size - w)
        pad_h = max(0, patch_size - h)
        img = transforms.functional.pad(img, padding=(0, 0, pad_w, pad_h), fill=0)
        w, h = img.size

    xs = list(range(0, max(1, w - patch_size + 1), stride))
    ys = list(range(0, max(1, h - patch_size + 1), stride))

    # Make sure the last patch always reaches the image border
    if xs[-1] != w - patch_size:
        xs.append(w - patch_size)
    if ys[-1] != h - patch_size:
        ys.append(h - patch_size)

    patches = []
    for y in ys:
        for x in xs:
            patches.append(img.crop((x, y, x + patch_size, y + patch_size)))

    return patches

def crop_and_apply_mask(img_path, mask_path, padding_ratio=0.6, min_size=128, apply_mask=True):
    """
    Crops the image around the foreground region defined by the mask.
    An additional padding is added to keep some context.
    Optionally applies the mask to remove background pixels.
    """
    img = Image.open(img_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")

    mask_arr = np.array(mask)
    ys, xs = np.where(mask_arr > 0)

    # If the mask is empty, return the original image
    if len(xs) == 0 or len(ys) == 0:
        return img

    # Bounding box of the foreground
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    h, w = mask_arr.shape
    pad_x = int((x_max - x_min + 1) * padding_ratio)
    pad_y = int((y_max - y_min + 1) * padding_ratio)

    # Expand bounding box with padding
    x_min = max(0, x_min - pad_x)
    y_min = max(0, y_min - pad_y)
    x_max = min(w - 1, x_max + pad_x)
    y_max = min(h - 1, y_max + pad_y)

    # Enforce a minimum crop size
    if (x_max - x_min + 1) < min_size:
        extra = (min_size - (x_max - x_min + 1)) // 2
        x_min = max(0, x_min - extra)
        x_max = min(w - 1, x_max + extra)

    if (y_max - y_min + 1) < min_size:
        extra = (min_size - (y_max - y_min + 1)) // 2
        y_min = max(0, y_min - extra)
        y_max = min(h - 1, y_max + extra)

    img_c = img.crop((x_min, y_min, x_max + 1, y_max + 1))

    # If masking is disabled, return only the cropped image
    if not apply_mask:
        return img_c

    # Apply binary mask to remove background
    mask_c = mask.crop((x_min, y_min, x_max + 1, y_max + 1))
    img_np = np.array(img_c).astype(np.float32)
    m_np = (np.array(mask_c) > 0).astype(np.float32)

    img_np = img_np * m_np[..., None]
    img_np = img_np.clip(0, 255).astype(np.uint8)

    return Image.fromarray(img_np)

def collate_patches(batch):
    """
    Custom collate function for the test loader.
    Each sample returns a variable number of patches, so we keep them as a list
    instead of stacking into a single tensor batch.
    """
    patch_tensors = [b[0] for b in batch]   # each element is [K, 3, H, W]
    sample_ids = [b[1] for b in batch]
    return patch_tensors, sample_ids

class BaseImageDataset(Dataset):
    """
    Training/validation dataset.
    Returns a single image tensor and its label.
    If use_mask_crop=True, it first crops around the mask and then samples one random patch.
    """
    def __init__(self, df, transform=None, use_mask_crop=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_mask_crop = use_mask_crop

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        label_idx = int(row["label_idx"])

        if self.use_mask_crop:
            mask_path = row["mask_path"]
            img = crop_and_apply_mask(img_path, mask_path)
            img = random_patch_pil(img, PATCH_SIZE)  # training uses a random patch
        else:
            img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        label = torch.tensor(label_idx, dtype=torch.long)
        return img, label


class TestImageDataset(Dataset):
    """
    Test dataset.
    Returns a stack of patches for each image (shape: [K, 3, H, W]) plus the sample id.
    """
    def __init__(self, df, transform=None, use_mask_crop=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_mask_crop = use_mask_crop

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        sample_id = row["sample_index"]

        if self.use_mask_crop:
            if self.transform is None:
                raise ValueError("TestImageDataset: transform cannot be None when use_mask_crop=True")

            mask_path = row["mask_path"]
            img = crop_and_apply_mask(img_path, mask_path)

            # Dense patch extraction for inference (overlap helps robustness)
            patches = grid_patches_pil(img, patch_size=PATCH_SIZE, stride=PATCH_SIZE // 2)

            patch_tensors = [self.transform(p) for p in patches]
            patch_tensors = torch.stack(patch_tensors, dim=0)  # [K, 3, H, W]
            return patch_tensors, sample_id

        # Fallback: no masking/patching, just load the image
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, sample_id


def make_loader(ds, batch_size, shuffle, drop_last):
    """
    Small wrapper around DataLoader with sane defaults for Colab.
    Note: prefetch_factor only works when num_workers > 0.
    """
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))

    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,
        pin_memory_device="cuda" if torch.cuda.is_available() else "",
    )

    if num_workers > 0:
        kwargs["prefetch_factor"] = 4

    return DataLoader(ds, **kwargs)

def compute_full_metrics(y_true, y_pred, y_prob, average="weighted"):
    """
    y_true: np.array shape [N]
    y_pred: np.array shape [N]
    y_prob: np.array shape [N, C] (probabilità softmax)
    """
    metrics = {}
    metrics["acc"]  = accuracy_score(y_true, y_pred)
    metrics["prec"] = precision_score(y_true, y_pred, average=average, zero_division=0)
    metrics["rec"]  = recall_score(y_true, y_pred, average=average, zero_division=0)
    metrics["f1"]   = f1_score(y_true, y_pred, average=average)

    # ROC AUC multi-classe: serve y_prob
    # Se per qualche ragione non è calcolabile (edge case), mettiamo NaN invece di crashare.
    try:
        metrics["roc_auc_ovr_weighted"] = roc_auc_score(
            y_true, y_prob, multi_class="ovr", average="weighted"
        )
    except Exception:
        metrics["roc_auc_ovr_weighted"] = float("nan")

    return metrics

def build_model(num_classes=4):
    """
    ConvNeXt-Base backbone pretrained on ImageNet.
    We replace the final classification head to match the challenge classes.
    """
    model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)

    # Fine-tune the whole network (no freezing)
    for p in model.parameters():
        p.requires_grad = True

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)

    return model

def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda=0, l2_lambda=0):
    """
    Perform one complete training epoch through the entire training dataset.

    Args:
        model (nn.Module): Neural network model to be trained.
        train_loader (DataLoader): PyTorch DataLoader providing (inputs, targets) batches.
        criterion (nn.Module): Loss function (e.g., CrossEntropyLoss, MSELoss).
        optimizer (torch.optim.Optimizer): Optimization algorithm.
        scaler (GradScaler): Gradient scaler for mixed precision training.
        device (torch.device): Computation device (CPU or GPU).
        l1_lambda (float): L1 regularization coefficient.
        l2_lambda (float): L2 regularization coefficient.

    Returns:
        tuple: (average_loss, f1_score) for the current epoch.
    """
    # Enable training mode (e.g., dropout and batch normalization)
    model.train()

    running_loss = 0.0           # Accumulates total loss over the epoch
    all_predictions = []         # Stores predictions for metric computation
    all_targets = []             # Stores ground-truth labels

    # Iterate over training batches
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to the selected device
        inputs, targets = inputs.to(device), targets.to(device)

        # Reset gradients to avoid accumulation from previous steps
        optimizer.zero_grad(set_to_none=True)

        # Forward pass with automatic mixed precision (GPU only)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(inputs)
            loss = criterion(logits, targets)

            # Optional L1 / L2 regularization (disabled by default)
            """
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm
            """

        # Backward pass with gradient scaling to prevent FP16 underflow
        scaler.scale(loss).backward()

        # Apply optimizer step if gradients are valid
        scaler.step(optimizer)

        # Update scaling factor for the next iteration
        scaler.update()

        # Accumulate batch loss weighted by batch size
        running_loss += loss.item() * inputs.size(0)

        # Compute class predictions from raw logits
        predictions = logits.argmax(dim=1)

        # Store results on CPU for metric computation
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Compute average loss over the entire dataset
    epoch_loss = running_loss / len(train_loader.dataset)

    # Compute weighted F1-score for the epoch
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average='weighted'
    )

    return epoch_loss, epoch_f1

def validate_one_epoch(model, val_loader, criterion, device, num_classes=4):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(inputs)
                loss = criterion(logits, targets)
                probs = F.softmax(logits, dim=1)

            running_loss += loss.item() * inputs.size(0)

            preds = logits.argmax(dim=1)

            all_targets.append(targets.detach().cpu().numpy())
            all_preds.append(preds.detach().cpu().numpy())
            all_probs.append(probs.detach().cpu().numpy())

    val_loss = running_loss / len(val_loader.dataset)

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_preds)
    y_prob = np.concatenate(all_probs)  # [N, C]

    metrics = compute_full_metrics(y_true, y_pred, y_prob, average="weighted")

    return val_loss, metrics, y_true, y_pred, y_prob

import json
from pathlib import Path

def fit(
    model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
    l1_lambda=0, l2_lambda=0, patience=0, evaluation_metric="val_f1", mode="max",
    restore_best_weights=True, writer=None, verbose=1, experiment_name="",
    num_classes=4
):
    history = {
        "train_loss": [], "val_loss": [],
        "train_f1": [],   "val_f1":   [],
        "val_acc": [], "val_prec": [], "val_rec": [], "val_roc_auc": []
    }

    best_epoch_targets = None
    best_epoch_preds = None
    best_epoch_probs = None

    Path("models").mkdir(exist_ok=True)

    # early stopping setup
    best_metric = float("-inf") if mode == "max" else float("inf")
    best_epoch = 0
    patience_counter = 0

    # path "best" sempre aggiornato (comodo per inference)
    best_ckpt_path = f"models/{experiment_name}_BEST.pt"
    best_json_path = f"models/{experiment_name}_BEST_metrics.json"

    print(f"Training {epochs} epochs...")

    for epoch in range(1, epochs + 1):
        train_loss, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
        )

        val_loss, val_metrics, y_true, y_pred, y_prob = validate_one_epoch(
            model, val_loader, criterion, device, num_classes=num_classes
        )

        # history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_f1"].append(train_f1)

        history["val_f1"].append(val_metrics["f1"])
        history["val_acc"].append(val_metrics["acc"])
        history["val_prec"].append(val_metrics["prec"])
        history["val_rec"].append(val_metrics["rec"])
        history["val_roc_auc"].append(val_metrics["roc_auc_ovr_weighted"])

        # print
        if verbose > 0 and (epoch % verbose == 0 or epoch == 1):
            print(
                f"Epoch {epoch:3d}/{epochs} | "
                f"Train: Loss={train_loss:.4f}, F1={train_f1:.4f} | "
                f"Val: Loss={val_loss:.4f}, "
                f"Acc={val_metrics['acc']:.4f}, "
                f"Prec={val_metrics['prec']:.4f}, "
                f"Rec={val_metrics['rec']:.4f}, "
                f"F1={val_metrics['f1']:.4f}, "
                f"ROC_AUC={val_metrics['roc_auc_ovr_weighted']:.4f}"
            )

        # metrica di selezione (di default usi val_f1)
        current_metric = val_metrics["f1"] if evaluation_metric == "val_f1" else history[evaluation_metric][-1]
        improved = (current_metric > best_metric) if mode == "max" else (current_metric < best_metric)

        if improved:
            best_metric = current_metric
            best_epoch = epoch
            best_epoch_targets = y_true
            best_epoch_preds = y_pred
            best_epoch_probs = y_prob

            # 1) salva checkpoint "versionato" con F1 nel nome
            f1_tag = f"{val_metrics['f1']:.4f}".replace(".", "_")
            ckpt_versioned = f"models/{experiment_name}_epoch{epoch:03d}_f1{f1_tag}.pt"
            torch.save(model.state_dict(), ckpt_versioned)

            # 2) aggiorna sempre anche un BEST fisso (comodo per inference)
            torch.save(model.state_dict(), best_ckpt_path)

            # 3) salva metrics in JSON (sia versionato che BEST)
            payload = {
                "experiment": experiment_name,
                "epoch": epoch,
                "train_loss": float(train_loss),
                "train_f1": float(train_f1),
                "val_loss": float(val_loss),
                "val_metrics": {k: float(v) for k, v in val_metrics.items()},
                "ckpt_versioned": ckpt_versioned,
                "ckpt_best": best_ckpt_path,
            }

            json_versioned = f"models/{experiment_name}_epoch{epoch:03d}_f1{f1_tag}_metrics.json"
            with open(json_versioned, "w") as f:
                json.dump(payload, f, indent=2)

            with open(best_json_path, "w") as f:
                json.dump(payload, f, indent=2)

            print(f"  ✅ New best! Saved: {ckpt_versioned}")
            patience_counter = 0
        else:
            if patience > 0:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping triggered at epoch {epoch}. Best epoch: {best_epoch} ({best_metric:.4f})")
                    break

    # restore best weights
    if restore_best_weights and best_epoch > 0:
        model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
        print(f"Restored BEST model from epoch {best_epoch} (best_metric={best_metric:.4f}).")

    if writer is not None:
        writer.close()

    return model, history, best_epoch_targets, best_epoch_preds, best_epoch_probs

# Hyperparameters
BATCH_SIZE = 16
EPOCHS = 50
PATIENCE = 12
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

X = train_df
y = train_df["label_idx"].values
fold_histories = []
fold_best_f1 = []
idx2label = data.idx2label
# checkpoints for the final ensemble (one per fold)
fold_ckpts = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(train_df, train_df["label_idx"].values), start=1
):
    print(f"\n========== FOLD {fold}/{N_SPLITS} ==========")

    # keep memory clean between folds (especially on GPU)
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    gc.collect()

    # split current fold
    train_df_split = train_df.iloc[train_idx].reset_index(drop=True)
    val_df_split   = train_df.iloc[val_idx].reset_index(drop=True)

    print("Train size:", len(train_df_split))
    print("Val size  :", len(val_df_split))

    # compute class weights from THIS fold (helps if classes are imbalanced)
    class_counts = train_df_split["label_idx"].value_counts().sort_index()
    class_weights = 1.0 / class_counts.values.astype(float)
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # datasets + loaders
    train_ds = BaseImageDataset(train_df_split, transform=train_transform)
    val_ds   = BaseImageDataset(val_df_split,   transform=val_transform)

    train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
    val_loader   = make_loader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    # fresh model for each fold (no leakage)
    model = build_model(num_classes=4).to(device)

    # separate backbone/head params to use different learning rates
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        (head_params if "classifier" in name else backbone_params).append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": backbone_params, "lr": 3e-5},
            {"params": head_params,     "lr": 3e-4},
        ],
        weight_decay=2e-4
    )

    # CrossEntropy with class weighting + a bit of label smoothing
    criterion = nn.CrossEntropyLoss(
        weight=class_weights_tensor,
        label_smoothing=0.05
    )

    # AMP scaler (enabled only on CUDA)
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    # train this fold (fit() will early-stop + restore best if you left defaults)
    cnn_model, training_history, all_targets, all_preds, all_probs = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=EPOCHS,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
        writer=None,
        verbose=1,
        experiment_name=f"cnn_fold{fold}",
        l2_lambda=0.0,
        patience=PATIENCE
    )

    fold_histories.append(training_history)
    fold_best_f1.append(max(training_history["val_f1"]))
    print(f"Best val F1 fold {fold}: {fold_best_f1[-1]:.4f}")

    # ----------------- training curves (this fold) -----------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))

    ax1.plot(training_history["train_loss"], label="Train loss", alpha=0.4, linestyle="--")
    ax1.plot(training_history["val_loss"],   label="Val loss",   alpha=0.9)
    ax1.set_title(f"Fold {fold} - Loss")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(training_history["train_f1"], label="Train F1", alpha=0.4, linestyle="--")
    ax2.plot(training_history["val_f1"],   label="Val F1",   alpha=0.9)
    ax2.set_title(f"Fold {fold} - F1")
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ---------------- confusion matrix (this fold) -----------------
    val_preds, val_targets = [], []

    cnn_model.eval()
    with torch.inference_mode():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            logits = cnn_model(xb)
            preds = logits.argmax(dim=1).cpu().numpy()
            val_preds.append(preds)
            val_targets.append(yb.numpy())

    val_preds = np.concatenate(val_preds)
    val_targets = np.concatenate(val_targets)

    val_acc  = accuracy_score(val_targets, val_preds)
    val_prec = precision_score(val_targets, val_preds, average="weighted", zero_division=0)
    val_rec  = recall_score(val_targets, val_preds, average="weighted", zero_division=0)
    val_f1m  = f1_score(val_targets, val_preds, average="weighted")

    cm = confusion_matrix(val_targets, val_preds)

    plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix — Fold {fold}")
    plt.tight_layout()
    plt.show()
    # ---------------------------------------------------------------

    target_names = [idx2label[i] for i in range(len(idx2label))]
    print("\nPer-class report(VAL)")
    print(classification_report(val_targets, val_preds, target_names = target_names, digits=3))
    print("Pred distribution:", np.bincount(all_preds, minlength=4))
    print("True distribution:", np.bincount(all_targets, minlength=4))


    # ---------------------------------------------------------------
    # save fold weights for ensemble inference later
    ckpt_path = f"models/ensemble_fold{fold}.pt"
    torch.save(cnn_model.state_dict(), ckpt_path)
    fold_ckpts.append(ckpt_path)
    print("Saved:", ckpt_path)

    # free everything before starting the next fold
    del model, optimizer, criterion, scaler, train_loader, val_loader, train_ds, val_ds, cnn_model
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print("\n=========== CV RESULTS ===========")
for i, f1 in enumerate(fold_best_f1, start=1):
    print(f"Fold {i}: best val F1 = {f1:.4f}")
print(f"Mean best F1: {np.mean(fold_best_f1):.4f} ± {np.std(fold_best_f1):.4f}")

test_ds = TestImageDataset(
    test_df,
    transform=val_transform,
    use_mask_crop=True
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_patches
)

NUM_CLASSES = 4
USE_TTA = True
TOP_K = 12
T = 1.0


MODEL_NAME = "cnn_fold1_epoch027_f10_4335.pt"
MODEL_CKPT = os.path.join("models", MODEL_NAME)
assert os.path.exists(MODEL_CKPT), f"Checkpoint non trovato: {MODEL_CKPT}"
print("Using model:", MODEL_CKPT)

model = build_model(num_classes=NUM_CLASSES).to(device)
model.load_state_dict(torch.load(MODEL_CKPT, map_location=device))
model.eval()


if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    model = model.to(memory_format=torch.channels_last)

@torch.inference_mode()
def infer_one_image_patches_safe(model, patches):
    if patches.is_cuda:
        patches = patches.contiguous(memory_format=torch.channels_last)

    views = [patches]
    if USE_TTA:
        views.append(torch.flip(patches, dims=[3]))
        views.append(torch.flip(patches, dims=[2]))

    V = len(views)
    K = patches.shape[0]
    C = NUM_CLASSES

    batch = torch.cat(views, dim=0)

    logits = model(batch) / T
    probs = F.softmax(logits, dim=1).view(V, K, C)

    conf = probs.max(dim=2).values
    k = min(TOP_K, K)
    topk_conf, topk_idx = conf.topk(k, dim=1)

    w = topk_conf / (topk_conf.sum(dim=1, keepdim=True) + 1e-12)
    gather_idx = topk_idx.unsqueeze(-1).expand(-1, -1, C)
    topk_probs = probs.gather(dim=1, index=gather_idx)

    probs_view = (topk_probs * w.unsqueeze(-1)).sum(dim=1)
    probs_img = probs_view.mean(dim=0)

    return probs_img


# --------- INFERENCE ---------
all_test_preds, all_test_ids = [], []

with torch.inference_mode():
    for patch_list, sample_ids in test_loader:
        for patches, sid in zip(patch_list, sample_ids):
            patches = patches.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, enabled=(device.type=="cuda")):
                probs = infer_one_image_patches_safe(model, patches)

            all_test_preds.append(int(probs.argmax()))
            all_test_ids.append(sid)

all_test_preds = np.array(all_test_preds)

# ==========================================

idx2label = data.idx2label


submission_ids = [
    f"{img_id}.png" if not str(img_id).endswith(".png") else str(img_id)
    for img_id in all_test_ids
]


all_test_labels = [idx2label[int(c)] for c in all_test_preds]


submission_df = pd.DataFrame({
    "sample_index": submission_ids,
    "label": all_test_labels
})

submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())